In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
from scripts.preprocessing.pipeline import process
from scripts.preprocessing.dqc_after_etl import validate_after_etl

from pprint import pprint
import pandas as pd

# Инфа ДО/ПОСЛЕ

In [3]:
train, (sample_submission, test), dqc_report = process(
    data_dir="../data", 
    save_report_path="report.json"
)

Обнаружено 4023 записей с дублирующимися названиями после чистки!
Примеры дубликатов:
    item_id                      item_name
12       12  МИХЕЙ И ДЖУМАНДЖИ Сука любовь
30       30        007 КООРДИНАТЫ СКАЙФОЛЛ
31       31        007 КООРДИНАТЫ СКАЙФОЛЛ
32       32                             11
33       33                             11
35       35                  10 ЛЕТ СПУСТЯ
36       36                  10 ЛЕТ СПУСТЯ
37       37                  10 ЛЕТ СПУСТЯ
71       71            11 ДРУЗЕЙ ОУШЕНА WB
72       72            11 ДРУЗЕЙ ОУШЕНА WB

Найдено 1657 уникальных названий, которые повторяются:
                       item_name  count
0        007 КООРДИНАТЫ СКАЙФОЛЛ      2
1                  10 ЛЕТ СПУСТЯ      3
2                             11      2
3            11 ДРУЗЕЙ ОУШЕНА WB      2
4            12 ДРУЗЕЙ ОУШЕНА WB      2
5                 12 ЛЕТ РАБСТВА      2
6                      127 ЧАСОВ      3
7                   12ДВЕНАДЦАТЬ      2
8            13 ДРУЗЕЙ ОУ

In [4]:
pprint(dqc_report)

{'categories': {'duplicate_ids': 0,
                'empty_names': 0,
                'n_missing': {'item_category_id': 0, 'item_category_name': 0},
                'n_rows': 84,
                'name': 'categories'},
 'items': {'bad_category_fk': 0,
           'duplicate_item_id': 0,
           'empty_names': 0,
           'missing': {'item_category_id': 0, 'item_id': 0, 'item_name': 0},
           'n_rows': 22170,
           'name': 'items'},
 'merge_quality': {'has_issues': False,
                   'missing_categories_count': 0,
                   'missing_items_count': 0,
                   'missing_shops_count': 0,
                   'null_columns_count': 0,
                   'null_counts': {'date': 0,
                                   'date_block_num': 0,
                                   'item_category_id': 0,
                                   'item_category_name': 0,
                                   'item_cnt_day': 0,
                                   'item_id': 0,
    

In [5]:
validate_after_etl(train)

In [6]:
print("Все проверки пройдены")
print(f"Train shape: {train.shape}")
print("\nПервые 25 строк train:\n")
train.head(25)

Все проверки пройдены
Train shape: (2935821, 10)

Первые 25 строк train:



,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,item_name,item_category_id,item_category_name,shop_name
0,2013-01-01,0,2,991,99.0,1.0,3D Action Puzzle Динозавры Тиранозавр,67,Подарки - Развитие,Адыгея ТЦ Мега
1,2013-01-01,0,2,1472,2599.0,1.0,Assassins Creed 3 Xbox 360 русская версия,23,Игры - XBOX 360,Адыгея ТЦ Мега
2,2013-01-01,0,2,1905,249.0,1.0,Bestseller. Grand Theft Auto San Andreas PC Jewel,30,Игры PC - Стандартные издания,Адыгея ТЦ Мега
3,2013-01-01,0,2,2920,599.0,2.0,Disney. LEGO Пираты Карибского моряPSP русская...,21,Игры - PSP,Адыгея ТЦ Мега
4,2013-01-01,0,2,3320,1999.0,1.0,FIFA 13PS3 русская версия,19,Игры - PS3,Адыгея ТЦ Мега
5,2013-01-01,0,2,4464,599.0,1.0,Lego Batman PS3,19,Игры - PS3,Адыгея ТЦ Мега
6,2013-01-01,0,2,4724,1399.0,1.0,Mafia IIXbox 360 русская версия,23,Игры - XBOX 360,Адыгея ТЦ Мега
7,2013-01-01,0,2,5649,2190.0,1.0,PS3 Файтстик Hori Mini 3,2,Аксессуары - PS3,Адыгея ТЦ Мега
8,2013-01-01,0,2,6911,599.0,1.0,Tekken 6PSP русская версия,21,Игры - PSP,Адыгея ТЦ Мега
9,2013-01-01,0,2,6916,999.5,1.0,Tekken Tag Tournament 2PS3 русские субтитры,19,Игры - PS3,Адыгея ТЦ Мега


# АНАЛИЗ ИЗНАЧАЛЬНЫХ ДАННЫХ

In [7]:
from scripts.preprocessing.load_data import get_data
item_categories_raw, items_raw, sales_train_raw, shops_raw, _, _ = get_data("../data")

In [8]:
print("=" * 60)
print("1. АНАЛИЗ ОТРИЦАТЕЛЬНЫХ ЦЕН")
print("=" * 60)

neg_price_rows = sales_train_raw[sales_train_raw['item_price'] < 0]
print(f"Найдено записей с отрицательной ценой: {len(neg_price_rows)}")
display(neg_price_rows)

if len(neg_price_rows) > 0:
    bad_item = neg_price_rows['item_id'].iloc[0]
    item_info = items_raw[items_raw['item_id'] == bad_item]
    print(f"\nИнформация о товаре с ID {bad_item}:")
    display(item_info)

1. АНАЛИЗ ОТРИЦАТЕЛЬНЫХ ЦЕН
Найдено записей с отрицательной ценой: 1


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
484683,15.05.2013,4,32,2973,-1.0,1.0



Информация о товаре с ID 2973:


,item_name,item_id,item_category_id
2973,"DmC Devil May Cry [PS3, русские субтитры]",2973,19


Причина:
- Возврат товара, оформленный как отрицательная продажа
- Ошибка при вводе данных (пропущен минус перед ценой?)

Как обрабатывать:
- Замена на медианную цену этого товара (item_median)
- Если у товара нет истории продаж — берём глобальную медиану

После обработки отрицательных цен не осталось

In [19]:
print("\n" + "=" * 60)
print("2. АНАЛИЗ ДУБЛИКАТОВ")
print("=" * 60)

duplicates = sales_train_raw[sales_train_raw.duplicated(subset=['date', 'shop_id', 'item_id'], keep=False)]
duplicates_sorted = duplicates.sort_values(['date', 'shop_id', 'item_id'])

print(f"Найдено записей с дубликатами по ключу (date, shop_id, item_id): {len(duplicates)}")
print(f"Из них уникальных групп дубликатов: {len(duplicates_sorted.groupby(['date', 'shop_id', 'item_id']))}")
print("\nПример дубликатов (первые 10):")
display(duplicates_sorted.head(10))


2. АНАЛИЗ ДУБЛИКАТОВ
Найдено записей с дубликатами по ключу (date, shop_id, item_id): 56
Из них уникальных групп дубликатов: 28

Пример дубликатов (первые 10):


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
1671872,01.05.2014,16,50,3423,999.0,1.0
1671873,01.05.2014,16,50,3423,999.0,1.0
284371,02.03.2013,2,16,12133,889.0,1.0
284372,02.03.2013,2,16,12133,1389.0,1.0
76961,05.01.2013,0,54,20130,149.0,1.0
76962,05.01.2013,0,54,20130,149.0,1.0
408103,06.04.2013,3,54,14050,198.0,1.0
408104,06.04.2013,3,54,14050,349.0,1.0
275474,07.03.2013,2,50,12133,1389.0,1.0
275476,07.03.2013,2,50,12133,889.0,1.0


Вероятная причина:
- Один и тот же чек пробит дважды
- Дневной отчёт разбит на несколько файлов с пересечениями
- Технический сбой при записи продаж

Способ обработки:
- Полные дубликаты (все колонки одинаковые) → удаляем (drop_duplicates)
- Дубликаты по ключу с разными item_cnt_day → суммируем продажи, цену берём first

После обработки дубликатов по ключу не осталось

In [15]:
print("\n" + "=" * 60)
print("3. АНАЛИЗ ВЫБРОСОВ")
print("=" * 60)

price_99 = sales_train_raw['item_price'].quantile(0.99)
cnt_99 = sales_train_raw['item_cnt_day'].quantile(0.99)

price_outliers = sales_train_raw[sales_train_raw['item_price'] > price_99]
cnt_outliers = sales_train_raw[sales_train_raw['item_cnt_day'] > cnt_99]

print(f"Ценовой порог (99-й перцентиль): {price_99:.2f}")
print(f"Количество ценовых выбросов: {len(price_outliers)}")
print(f"\nПримеры товаров с аномально высокой ценой:")
display(price_outliers.head(5))

print(f"\nКоличество порог (99-й перцентиль): {cnt_99:.2f}")
print(f"Количество выбросов по количеству: {len(cnt_outliers)}")
print(f"\nПримеры аномально больших продаж в один день:")
display(cnt_outliers.head(5))


3. АНАЛИЗ ВЫБРОСОВ
Ценовой порог (99-й перцентиль): 5999.00
Количество ценовых выбросов: 28714

Примеры товаров с аномально высокой ценой:


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
1066,18.01.2013,0,25,4861,8490.0,1.0
1203,20.01.2013,0,25,5613,6190.0,1.0
1701,03.01.2013,0,25,4384,13499.0,1.0
1949,05.01.2013,0,24,5371,8290.0,1.0
1950,24.01.2013,0,24,5371,7590.0,1.0



Количество порог (99-й перцентиль): 5.00
Количество выбросов по количеству: 27411

Примеры аномально больших продаж в один день:


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
352,15.01.2013,0,25,2973,2499.0,13.0
404,25.01.2013,0,25,2972,599.0,13.0
3238,19.01.2013,0,25,32,349.0,7.0
3252,26.01.2013,0,25,35,399.0,6.0
3784,20.01.2013,0,25,14346,399.0,8.0


Причина:
- Для цен: дорогая техника, коллекционные товары, комплекты
- Для количества: оптовые закупки, праздничный ажиотаж или уникальные события(услуги)

Способ обработки:
- Выбросы НЕ удаляются и НЕ обрезаются, так как могут содержать важную информацию
- Будем применять log-трансформацию перед подачей в модель

In [13]:
print("\n" + "=" * 60)
print("4. АНАЛИЗ 'ХОЛОДНЫХ' ТОВАРОВ")
print("=" * 60)

items_with_one_block = sales_train_raw.groupby('item_id')['date_block_num'].nunique()
cold_items = items_with_one_block[items_with_one_block == 1]
print(f"Товаров, продававшихся только в одном месяце: {len(cold_items)}")
print(f"Это {len(cold_items)/len(items_raw)*100:.1f}% от всех товаров")

print("\nПримеры таких товаров:")
cold_items_examples = sales_train_raw[sales_train_raw['item_id'].isin(cold_items.head(5).index)]
display(cold_items_examples[['item_id', 'date_block_num', 'item_cnt_day']].head(10))


4. АНАЛИЗ 'ХОЛОДНЫХ' ТОВАРОВ
Товаров, продававшихся только в одном месяце: 2999
Это 13.5% от всех товаров

Примеры таких товаров:


,item_id,date_block_num,item_cnt_day
1812428,6,18,1.0
1972624,4,20,1.0
1972636,0,20,1.0
2280989,7,23,1.0
2280990,5,23,1.0


Причина:
- Сезонные товары (новогодние украшения, школьные принадлежности)
- Новинки, которые перестали завозить
- Акционные товары, которые быстро закончились

Влияние на модель:
- Для таких товаров нельзя использовать собственные лаги, так как нету истории
- Придётся опираться на статистику по категории или магазину

In [16]:
print("\n" + "=" * 60)
print("5. ИТОГОВАЯ ТАБЛИЦА: ОБНАРУЖЕНО VS ОБРАБОТАНО")
print("=" * 60)

summary_table = pd.DataFrame({
    'Проблема': [
                'Отрицательные цены', 
                'Дубликаты строк (полные)', 
                'Дубликаты по ключу (date, shop_id, item_id)',
                'Ценовые выбросы (>99%)',
                'Выбросы по количеству (>99%)',
                'Товары с продажами в 1 месяце'
                ],
    'Обнаружено': [
                dqc_report['sales']['negative_price'],
                dqc_report['sales']['duplicate_rows'],
                dqc_report['sales']['duplicate_keys'],
                dqc_report['sales']['price_outliers'],
                dqc_report['sales']['cnt_outliers'],
                dqc_report['sales']['items_with_one_block']
                ],
    'Обработка': [
                'Замена на медиану товара',
                'Удаление (drop_duplicates)',
                'Агрегация (sum + first)',
                'Оставлены как есть',
                'Оставлены как есть',
                'Будут использованы category-фичи'
                ],
    'Результат': [
                '0 осталось',
                '0 осталось',
                '0 осталось',
                f'{dqc_report["sales"]["price_outliers"]} в данных',
                f'{dqc_report["sales"]["cnt_outliers"]} в данных',
                '—'
                ]
})

display(summary_table)


5. ИТОГОВАЯ ТАБЛИЦА: ОБНАРУЖЕНО VS ОБРАБОТАНО


,Проблема,Обнаружено,Обработка,Результат
0,Отрицательные цены,1,Замена на медиану товара,0 осталось
1,Дубликаты строк (полные),6,Удаление (drop_duplicates),0 осталось
2,"Дубликаты по ключу (date, shop_id, item_id)",28,Агрегация (sum + first),0 осталось
3,Ценовые выбросы (>99%),28714,Оставлены как есть,28714 в данных
4,Выбросы по количеству (>99%),27411,Оставлены как есть,27411 в данных
5,Товары с продажами в 1 месяце,2999,Будут использованы category-фичи,—
